# 03 — Vector Embeddings
**FootScout — BHT Berlin**

Mandatory WP: Statistical PCA/UMAP + text + hybrid embedding pipeline.


In [ ]:
import sys; sys.path.insert(0,"..")
import pandas as pd, numpy as np, wandb
from src.embeddings import (StatisticalEmbedder, TextEmbedder,
    build_hybrid_embedding, build_embeddings, load_embeddings)
df = pd.read_csv("../data/processed/players_merged.csv", low_memory=False)
print(f"{len(df):,} players")

## 1. Statistical Embedding — PCA Baseline

In [ ]:
stat_pca = StatisticalEmbedder(method="pca", n_components=32)
X_pca = stat_pca.fit_transform(df)
ev = stat_pca._reducer.explained_variance_ratio_.sum()
print(f"PCA shape: {X_pca.shape} | Explained variance: {ev:.3f}")

## 2. Statistical Embedding — UMAP (Recommended)

In [ ]:
stat_umap = StatisticalEmbedder(method="umap", n_components=32, umap_n_neighbors=15)
X_umap = stat_umap.fit_transform(df)
print(f"UMAP shape: {X_umap.shape}")

## 3. UMAP 2D Cluster Visualization

In [ ]:
import umap, plotly.express as px
from sklearn.preprocessing import StandardScaler
from src.embeddings import ALL_STAT_FEATURES
feat = [c for c in ALL_STAT_FEATURES if c in df.columns]
X_sc = StandardScaler().fit_transform(df[feat].apply(pd.to_numeric,errors="coerce").fillna(0))
emb2d = umap.UMAP(n_components=2,n_neighbors=15,random_state=42).fit_transform(X_sc)
df2 = df[["player","pos","league","squad"]].copy()
df2["x"]=emb2d[:,0]; df2["y"]=emb2d[:,1]
fig=px.scatter(df2,x="x",y="y",color="pos",hover_name="player",hover_data=["squad","league"],
    title="UMAP 2D — Player Style Clusters",template="plotly_dark")
fig.show()

## 4. Text Embedding — Sentence Transformers

In [ ]:
text_emb = TextEmbedder()
for i in [0,50,100]:
    print(f"Profile {i}:
{text_emb.build_profile(df.iloc[i])}
")
X_text = text_emb.encode(df)
print(f"Text embedding shape: {X_text.shape}")

## 5. Hybrid Embedding (alpha=0.6)

In [ ]:
ALPHA = 0.6
hybrid = build_hybrid_embedding(X_umap, X_text, alpha=ALPHA)
print(f"Hybrid shape: {hybrid.shape}")
run = wandb.init(project="footscout", name=f"emb_umap_a{int(ALPHA*100)}", job_type="embeddings")
results = build_embeddings(df, method="umap", n_components=32, alpha=ALPHA, wandb_run=run, save=True)
run.finish(); print("✅ Embeddings saved")